# ***Events Normalization (Loops)***

This notebook contains the code used to **normalize GA4 events data** by flattening the `event_params` and `user_properties` columns using loops.

In [1]:
import json
import pandas as pd
import time
import datetime
import orjson

In [2]:
with open('../../raw_data/events.json', 'r') as f:
    data = json.load(f)

In [3]:
data_df = pd.json_normalize(data,sep='_')

## Version 1 - Manula Normalization

In [4]:
params_data = [
        {
            item["key"]: next(v for v in item["value"].values() if v is not None )
            for item in params_list
        }
        for params_list in data_df["event_params"]
    ]
param_df = pd.DataFrame(params_data).add_prefix("ep_")

In [5]:
user_props_data = [
        {
            prop["key"]: next(
                v
                for k, v in prop["value"].items()
                if v is not None and k != "set_timestamp_micros"
            )
            for prop in props_list
        }
        for props_list in data_df["user_properties"]
    ]
user_props_df = pd.DataFrame(user_props_data).add_prefix("user_prop_")

In [6]:
final_df_v1 = pd.concat([data_df, param_df, user_props_df], axis=1)
final_df_v1.drop(['event_params', 'user_properties'], axis=1, inplace=True)
event_date_dt = pd.to_datetime(final_df_v1["event_date"], format="%Y%m%d")
final_df_v1['year']= event_date_dt.dt.year
final_df_v1['month'] = event_date_dt.dt.month
final_df_v1

,event_date,event_timestamp,event_name,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,user_first_touch_timestamp,...,ep_value,ep_link_text,ep_file_extension,ep_file_name,ep_link_url,ep_content,user_prop_user_session_id,user_prop_user_client_id,year,month
0,20241113,1731513971041603,first_visit,None,None,-2094340797,None,None,2091574202.1731513971,1.731514e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,11
1,20241113,1731513971041603,session_start,None,None,-2094340797,None,None,2091574202.1731513971,1.731514e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,11
2,20241113,1731513971041603,page_view,None,None,-2094340797,None,None,2091574202.1731513971,1.731514e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,11
3,20241113,1731513976070304,user_session_info,None,None,-2089312096,None,None,2091574202.1731513971,1.731514e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,_1731513970,_2091574202.1731513971,2024,11
4,20241113,1731513976070304,scroll,None,None,-2089312096,None,None,2091574202.1731513971,1.731514e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,_1731513970,_2091574202.1731513971,2024,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260859,20241113,1731552192524186,user_session_info,None,None,1767403418,None,None,1436230623.1728949201,1.728949e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,_1731552186,_1436230623.1728949201,2024,11
260860,20241113,1731552192524186,scroll,None,None,1767403418,None,None,1436230623.1728949201,1.728949e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,_1731552186,_1436230623.1728949201,2024,11
260861,20241113,1731552192524186,scroll,None,None,1767403418,None,None,1436230623.1728949201,1.728949e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,_1731552186,_1436230623.1728949201,2024,11
260862,20241113,1731552192524186,scroll,None,None,1767403418,None,None,1436230623.1728949201,1.728949e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,_1731552186,_1436230623.1728949201,2024,11


## Version 2 - Function-based Normalization

Single function for processing `event_params` and `user_properties`

In [7]:
def normalize_params(df,column_name,prefix):
    data = []
    for items_list in df[column_name]:
        row_dict = {
            item["key"]: next(
                v
                for k, v in item["value"].items()
                if v is not None and k != "set_timestamp_micros"
            )
            for item in items_list
        }
        data.append(row_dict)
    return pd.DataFrame(data).add_prefix(prefix)

In [8]:
event_parameters = normalize_params(data_df,"event_params","ep_")
event_parameters

,ep_batch_page_id,ep_source,ep_page_location,ep_engaged_session_event,ep_campaign,ep_batch_ordering_id,ep_page_title,ep_medium,ep_session_engaged,ep_ga_session_id,...,ep_click_url,ep_ignore_referrer,ep_click_classes,ep_click_id,ep_value,ep_link_text,ep_file_extension,ep_file_name,ep_link_url,ep_content
0,1731513969906,prm,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,productpickup,1,Eye Care Center in Crystal Lake - Shopko Optical,4pc,0,1731513970,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1731513969906,prm,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,productpickup,1,Eye Care Center in Crystal Lake - Shopko Optical,4pc,1,1731513970,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1731513969906,prm,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,productpickup,1,Eye Care Center in Crystal Lake - Shopko Optical,4pc,0,1731513970,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1731513969906,NaN,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,NaN,2,Eye Care Center in Crystal Lake - Shopko Optical,NaN,0,1731513970,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1731513969906,NaN,https://www.shopko.com/eye-care/il/crystal-lak...,1.0,NaN,2,Eye Care Center in Crystal Lake - Shopko Optical,NaN,0,1731513970,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260859,1731552185721,NaN,https://www.shopko.com/eye-care/?utm_source=em...,1.0,NaN,2,Eye Exam Center Locations Near You,NaN,0,1731552186,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
260860,1731552185721,NaN,https://www.shopko.com/eye-care/?utm_source=em...,1.0,NaN,2,Eye Exam Center Locations Near You,NaN,0,1731552186,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
260861,1731552185721,NaN,https://www.shopko.com/eye-care/?utm_source=em...,1.0,NaN,2,Eye Exam Center Locations Near You,NaN,0,1731552186,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
260862,1731552185721,NaN,https://www.shopko.com/eye-care/?utm_source=em...,1.0,NaN,2,Eye Exam Center Locations Near You,NaN,0,1731552186,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
user_properties = normalize_params(data_df,"user_properties","user_prop_")
user_properties

,user_prop_user_session_id,user_prop_user_client_id
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,_1731513970,_2091574202.1731513971
4,_1731513970,_2091574202.1731513971
...,...,...
260859,_1731552186,_1436230623.1728949201
260860,_1731552186,_1436230623.1728949201
260861,_1731552186,_1436230623.1728949201
260862,_1731552186,_1436230623.1728949201


In [10]:
final_df_v2 = pd.concat([data_df, event_parameters, user_properties], axis=1)
final_df_v2.drop(['event_params', 'user_properties'], axis=1, inplace=True)
event_date_dt = pd.to_datetime(final_df_v2["event_date"], format="%Y%m%d")
final_df_v2['year']= event_date_dt.dt.year
final_df_v2['month'] = event_date_dt.dt.month
final_df_v2

,event_date,event_timestamp,event_name,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,user_first_touch_timestamp,...,ep_value,ep_link_text,ep_file_extension,ep_file_name,ep_link_url,ep_content,user_prop_user_session_id,user_prop_user_client_id,year,month
0,20241113,1731513971041603,first_visit,None,None,-2094340797,None,None,2091574202.1731513971,1.731514e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,11
1,20241113,1731513971041603,session_start,None,None,-2094340797,None,None,2091574202.1731513971,1.731514e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,11
2,20241113,1731513971041603,page_view,None,None,-2094340797,None,None,2091574202.1731513971,1.731514e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2024,11
3,20241113,1731513976070304,user_session_info,None,None,-2089312096,None,None,2091574202.1731513971,1.731514e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,_1731513970,_2091574202.1731513971,2024,11
4,20241113,1731513976070304,scroll,None,None,-2089312096,None,None,2091574202.1731513971,1.731514e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,_1731513970,_2091574202.1731513971,2024,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
260859,20241113,1731552192524186,user_session_info,None,None,1767403418,None,None,1436230623.1728949201,1.728949e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,_1731552186,_1436230623.1728949201,2024,11
260860,20241113,1731552192524186,scroll,None,None,1767403418,None,None,1436230623.1728949201,1.728949e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,_1731552186,_1436230623.1728949201,2024,11
260861,20241113,1731552192524186,scroll,None,None,1767403418,None,None,1436230623.1728949201,1.728949e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,_1731552186,_1436230623.1728949201,2024,11
260862,20241113,1731552192524186,scroll,None,None,1767403418,None,None,1436230623.1728949201,1.728949e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,_1731552186,_1436230623.1728949201,2024,11
